# PWM Example 2 — Multiple channels and live duty updates

**Target:** STM32F103C8T6 · TIM2_CH1–CH3  
**Focus:** Share a PWM frequency while controlling independent outputs.

## Feature classification

| Update style | Configuration | Behavior |
|---|---|---|
| Static compare | Set CCR before start | Fixed duty cycle |
| Immediate update | Write CCR while running, preload disabled | New compare takes effect immediately; may produce a partial-cycle edge |
| Preloaded update | OC preload enabled | CCR shadow value transfers at update event, avoiding mid-cycle glitch |
| DMA waveform | Timer DMA burst / channel update | Sequenced duty values with reduced CPU load; buffer timing/ownership matters |
| Multiple channels | Same timer, one CCR per channel | Shared carrier frequency, separate duty values |

Default TIM2 outputs are CH1 PA0, CH2 PA1, CH3 PA2 and CH4 PA3. The repository `STM32F103C8T6_TIM_PWM_Output` starts CH1–CH3 and updates their compare registers. Channels on one timer share the same counter period and prescaler; they cannot have independent PWM frequencies. Contrast this with `STM32F103C8T6_leds_rgb_ex1`, which toggles GPIO instead of generating hardware PWM.

## Example — Three-channel RGB LED

Configure TIM2 CH1–CH3 with one common 1 kHz period and active-high PWM. Use external resistors and verify whether the RGB package is common-anode or common-cathode; common-anode usually requires inverted duty/polarity logic. Match the count range to the configured ARR.

```c
#define PWM_COUNTS 1000U

HAL_TIM_PWM_Start(&htim2, TIM_CHANNEL_1);
HAL_TIM_PWM_Start(&htim2, TIM_CHANNEL_2);
HAL_TIM_PWM_Start(&htim2, TIM_CHANNEL_3);

static void set_rgb(uint16_t red, uint16_t green, uint16_t blue)
{
    if (red > PWM_COUNTS) red = PWM_COUNTS;
    if (green > PWM_COUNTS) green = PWM_COUNTS;
    if (blue > PWM_COUNTS) blue = PWM_COUNTS;
    __HAL_TIM_SET_COMPARE(&htim2, TIM_CHANNEL_1, red);
    __HAL_TIM_SET_COMPARE(&htim2, TIM_CHANNEL_2, green);
    __HAL_TIM_SET_COMPARE(&htim2, TIM_CHANNEL_3, blue);
}
```


## Preload and clean transitions

Enable output-compare preload for each channel (CubeMX option or `OCxPE` via the HAL configuration). The next CCR value is latched at the timer update event. Auto-reload preload can similarly defer ARR changes. Coordinate updates across channels if a coherent color/phase change is needed; update event timing is common, but sequential CPU writes still occur at different instants.

Avoid changing PSC/ARR/CCR without understanding the update event. If the output must begin in a safe state, establish pin polarity and initial compare values before enabling the output channel.

## Practical variations

- Use PWM1 or PWM2 according to whether the active interval should occur before or after the compare point.
- Use active-low polarity for circuits that sink current, after checking the external circuit.
- Use one timer for synchronized multi-channel outputs; use separate timers only when independent periods are required and timing synchronization is not essential.
- For an RGB LED, human brightness perception is nonlinear; apply a gamma lookup table if perceptual brightness matters.
- Timer DMA is not automatically equivalent to DMA from every CCR on every update; configure the correct DMA request/channel and transfer width for the selected timer.